# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore a Croissant AI-ready dataset using the `mlcroissant` library and the FAIR² mental health survey dataset.

### Dataset Source
<ul>
  <li><b>FAIR² Croissant schema URL:</b> <a href="https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json">https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json</a></li>
</ul>


In [ ]:
# Ensure mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the FAIR² Mental Health Dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# View dataset metadata (name and description)
meta = dataset.metadata
print("Dataset name:", getattr(meta, 'name', None))
print("Description:")
print(getattr(meta, 'description', None))

## 2. Data Overview
Let's enumerate available record sets, their `@id`s, and their respective field (column) `@id`s, so we can choose which to explore.

Notes:
- All Croissant entities (record sets, fields/columns) are referenced by their `@id` field.
- We'll print these so you can reference them in further steps.

In [ ]:
# List all RecordSets (tables) and their fields, referenced by @id
print("Available `recordSet` entries in the dataset:")
if hasattr(meta, "record_set") and meta.record_set:
    for rs in meta.record_set:
        print(f"- RecordSet name: {getattr(rs, 'name', None)} | @id: {getattr(rs, '@id', None)}")
        print("    Columns (fields):")
        # Print fields (columns) for each record set
        if hasattr(rs, "field") and rs.field:
            for field in rs.field:
                print(f"      - {getattr(field, 'name', None):20} (@id: {getattr(field, '@id', None)})    type: {getattr(field, 'data_type', None)}")
        elif hasattr(rs, "column") and rs.column:  # Sometimes columns are called 'column'
            for col in rs.column:
                print(f"      - {getattr(col, 'name', None):20} (@id: {getattr(col, '@id', None)})    type: {getattr(col, 'data_type', None)}")
        else:
            print("      (No fields found)")
else:
    print("(No recordSet entities found in metadata.)")

## 3. Data Extraction
Now let's load data from one or more record sets into pandas DataFrames for analysis.

**Steps:**
1. Specify the list of record set `@id`s you want to extract.
2. Extract and load all records from each chosen record set.
3. Preview the columns and view a sample of the data.

📝 *Modify the `record_sets_ids` list below if you want to load more or other record sets from the dataset.*

In [ ]:
#-- Get available record set @ids (from above output or metadata programmatically) --#
record_set_ids = []

# Programmatically gather all record set @ids
if hasattr(meta, "record_set") and meta.record_set:
    record_set_ids = [getattr(rs, "@id", None) for rs in meta.record_set if getattr(rs, "@id", None)]

# We'll extract ALL recordsets (can restrict as desired)
print("Loading the following record set @ids:")
for rid in record_set_ids:
    print("  -", rid)

dataframes = {}
for rs_id in record_set_ids:
    print(f"\nExtracting records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print("  (No records found.)")
        continue
    df = pd.DataFrame(records)
    print("  Columns:", df.columns.tolist())
    print("  Sample:")
    display(df.head())
    dataframes[rs_id] = df
# For the rest of the notebook, we pick the main survey data RecordSet for further EDA
primary_record_set = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Let's filter and transform the data in the main record set.

Examples below (edit field names/IDs and logic for your own analysis):

- We'll use the GAD-7 total score column for numeric analysis, filtering on a threshold, and normalizing it.
- We'll also group data by gender if a gender field exists in the record set.

In [ ]:
import numpy as np

if primary_record_set is None or primary_record_set not in dataframes:
    print("No primary record set with data to analyze.")
else:
    df = dataframes[primary_record_set]
    
    # Find a typical GAD-7 total score field by searching columns containing 'gad' and 'total'
    numeric_field_id = None
    for c in df.columns:
        if "gad" in c.lower() and "total" in c.lower():
            numeric_field_id = c
            break
    # If not found, pick first numeric column as fallback
    if numeric_field_id is None:
        numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        numeric_field_id = numeric_fields[0] if numeric_fields else None
    print(f"Numeric field chosen for analysis: {numeric_field_id}")
    
    # Ensure column is numeric type
    if numeric_field_id is not None:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = 10  # Example GAD-7 moderate anxiety cutoff
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} records")
        display(filtered_df.head())
        # Z-score normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Attempt grouping by gender if present
        group_field_id = None
        for c in df.columns:
            if "gender" in c.lower():
                group_field_id = c
                break
        if group_field_id:
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Mean GAD-7 scores by group:")
            display(grouped_df)
        else:
            print("No group field (e.g., gender) found for grouping.")

## 5. Visualization
Let's plot distributions and group comparisons for the selected numeric field (if present) using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set is None or primary_record_set not in dataframes:
    print("No data to visualize.")
else:
    df = dataframes[primary_record_set]
    # Use determined numeric_field_id and group_field_id if exist
    numeric = None
    for c in df.columns:
        if "gad" in c.lower() and "total" in c.lower():
            numeric = c
            break
    if numeric is None:
        numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        numeric = numeric_fields[0] if numeric_fields else None
    print(f"Using {numeric} for visualization")
    if numeric and numeric in df.columns:
        df[numeric] = pd.to_numeric(df[numeric], errors='coerce')
        plt.figure(figsize=(8,6))
        sns.histplot(df[numeric].dropna(), bins=15, kde=True, color='skyblue')
        plt.title(f"Distribution of {numeric}")
        plt.xlabel(numeric)
        plt.show()
        
        # Boxplot by gender, if present
        group = None
        for c in df.columns:
            if "gender" in c.lower():
                group = c
                break
        if group and group in df.columns:
            plt.figure(figsize=(7,5))
            sns.boxplot(x=df[group], y=df[numeric], palette="Set2")
            plt.title(f"{numeric} by {group}")
            plt.xlabel(group)
            plt.ylabel(numeric)
            plt.show()

## 6. Conclusion
In this notebook, we've introduced the FAIR² mental health survey dataset, explored its schema (record sets and fields by `@id`), loaded data using `mlcroissant`, and performed basic exploratory analyses. You can adapt the approach above to analyze other fields, perform advanced cleaning, or build predictive models.

**Key Takeaways:**
- The Croissant format, together with `mlcroissant`, enables structured, standards-based AI-ready data loading and discovery.
- It's easy to enumerate available datasets, fields, and perform analysis—while always referencing by Croissant entity `@id`.

For further analysis, consult the dataset's documentation, or extend this notebook for predictive modeling or visual analytics.